# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**My lane:** Content Refresh Prioritization (from ML-02/ML-03).

**One row means:** one content item's Google Search Console performance for one calendar day,
for one client — the natural grain of `fact_content_daily_performance`
(`report_date + client_hash_id + content_hash_id`).

**Table(s) I use:** `fact_content_daily_performance` (the `month=2026-03` partition only —
never the `_sample` table, since that's the sealed final month). I join `dim_clients` once, to
filter on tracking availability.

**Time window:** March 2026 (`2026-03-01` → `2026-03-31`), a mid-panel month. I split it into a
first half (days 1–15, my "decision moment") and a second half (days 16–31, my outcome window).

**What I'd predict/rank (label or proxy):** `is_declining_proxy` — 1 if a content item's GSC
impressions in the second half of March are more than 20% lower than the first half, else 0.
This mirrors the starter dataset's `trend_direction` logic on purpose, and it is a **proxy**,
not a real future-outcome label (the guide is explicit that this kind of same-window bucket is
a beginner proxy, not the capstone-grade target) — a stronger version later would predict April
from March instead of second-half-March from first-half-March.

**One thing I deliberately exclude:** `gsc_avg_position` rows where the value is `0` — that
means "no position data," not rank zero, so I filter those out before averaging rather than
letting them silently drag every average toward "great position."


In [ ]:
%pip -q install duckdb

import os, getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt (last resort).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    # mid-panel month partition only — cheap, and never the sealed final month
    'fact_march':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

print('Connected. Ready to query the March 2026 partition.')


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `report_date` | Context | Defines the time window / split point — not something a model learns from |
| `client_hash_id` | Context | Join/grouping key only, per the field-types guide — never a feature |
| `content_hash_id` | Context | Join/grouping key only — the grain's other half |
| `gsc_impressions` (days 1–15) | Feature | Observed signal, known before the decision moment |
| `gsc_clicks` (days 1–15) | Feature | Observed signal, known before the decision moment |
| `gsc_avg_position` (days 1–15, filtered `> 0`) | Feature | Observed signal; `0` rows are excluded first (see below) |
| `ga4_data_available` | Context | A tracking-coverage flag, not a performance signal — used to filter, not to predict |
| `gsc_impressions` (days 16–31) | Label component | This is what `is_declining_proxy` is computed FROM — never a feature (the exact leakage trap in part 3) |
| `gsc_avg_position` where value `= 0` | Excluded | Means "no position data," not rank zero — would silently bias every average toward looking great |

I confirm the real column names below with `DESCRIBE`, rather than assuming them.


In [ ]:
cols = con.sql(f"DESCRIBE {TABLES['fact_march']}").df()
print(cols[['column_name', 'column_type']].to_string(index=False))


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check
One row really should be one `report_date + client_hash_id + content_hash_id`. If any
combination appears more than once, my stated grain is wrong.


In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'Rows with a duplicated grain: {len(grain_check)}')
grain_check


### 3b. Row count + date span
My slice's size and dates, so I can catch a wrong month or a truncated read early.


In [ ]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_march']}
""").df()

span


### 3c. Availability
GA4 availability is a three-valued flag (`TRUE` / `FALSE` / `NULL`) — `= FALSE` or `NOT ...`
mishandles the `NULL` rows silently, so I filter with `IS TRUE` / `IS NOT TRUE` as the skill
warns.


In [ ]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_unavailable_or_null_rows
    FROM {TABLES['fact_march']}
""").df()

availability['pct_ga4_available'] = (
    availability['ga4_available_rows'] / availability['total_rows'] * 100
).round(1)
availability


### 3d. Five features (max) — built from the first half of March only

Decision moment = end of day 15. Every feature below is computed **only** from
`report_date <= '2026-03-15'`, so every one of them is knowable at that moment.


In [ ]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                    AS impressions_first_half,
        SUM(gsc_clicks)                                         AS clicks_first_half,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_first_half,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_first_half,
        SUM(gsc_clicks) / GREATEST(SUM(gsc_impressions), 1)      AS ctr_first_half
    FROM {TABLES['fact_march']}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10   -- drop near-zero-volume noise before it enters the frame
""").df()

print(f'{len(features):,} content items with enough first-half volume')
features.head()


**Feature notes — "knowable at the decision moment because…"**

1. `impressions_first_half` — knowable because it only sums `report_date <= 2026-03-15`, entirely
   before the decision moment.
2. `clicks_first_half` — same window restriction; nothing here comes from the second half.
3. `avg_position_first_half` — knowable for the same reason, and I first filter out `= 0` rows
   (no position data) so a missing-data code doesn't masquerade as a great average.
4. `active_days_first_half` — a consistency signal (how many of the 15 days had any impressions
   at all), computed only over the first half.
5. `ctr_first_half` — a ratio of two first-half-only sums, so it inherits the same "before the
   decision moment" guarantee as its inputs.


### 3e. The trap — one label-derived column, on purpose

Now I build the label itself from the second half, join it to the honest feature frame, train a
quick classifier, and get an honest baseline score.


In [ ]:
label = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h1,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h2
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
""").df()

label['is_declining_proxy'] = (label['imp_h2'] < 0.8 * label['imp_h1']).astype(int)

data = features.merge(label[['client_hash_id', 'content_hash_id', 'imp_h2', 'is_declining_proxy']],
                       on=['client_hash_id', 'content_hash_id'], how='inner')
print(f'labeled rows: {len(data):,}, decline rate: {data["is_declining_proxy"].mean():.3f}')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_cols = ['impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
               'active_days_first_half', 'ctr_first_half']

def quick_score(df, cols):
    d = df.dropna(subset=cols + ['is_declining_proxy'])
    X, y = d[cols], d['is_declining_proxy']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_score(data, honest_cols)
print(f'Honest AUC (5 features only): {honest_auc:.3f}')

# --- THE TRAP: add ONE label-derived column on purpose ---
leaky_cols = honest_cols + ['imp_h2']   # imp_h2 is literally what the label is computed from
leaky_auc = quick_score(data, leaky_cols)
print(f'Leaky AUC (+ imp_h2, a label-derived column): {leaky_auc:.3f}')
print(f'Jump: {leaky_auc - honest_auc:+.3f}')


**What happened:** adding `imp_h2` — the second-half impressions the label is computed
directly from — pushes the AUC toward ~1.0, because the model isn't finding a pattern, it's
reading the answer key. This is the exact leakage lesson from notebook 02, just performed on
real warehouse data instead of the starter CSV: a feature computed from the same window (or a
component of) the label will always look like a great model and always be useless in production,
because on a real future day that column doesn't exist yet.

**I delete `imp_h2` and keep the honest number** — `honest_auc` above, from the 5 features that
were genuinely knowable at the decision moment.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** this is an **unbalanced panel** — per-client history depth differs
wildly (some clients have 17 months of history, some only 3), and I did not filter on
`dim_clients.gsc_data_start` before pulling March 2026. That means some "low activity" content
items in my feature frame might just belong to a client whose tracking started partway through
the panel, not to a client that is genuinely declining. A stronger version of this contract would
join `dim_clients` first and either restrict to clients with a full March of history, or add
`days_since_client_tracking_started` as a context column so a reviewer can tell the two cases
apart.

This data also cannot tell me *why* impressions moved — a real decline, a sibling page
absorbing demand (consolidation), plain seasonality, or a SERP/AI-click change with position
holding steady. Section 7 of the lane guide names these look-alikes; this notebook only proves
the mechanics of the contract, not which explanation is correct for any given page.


In [ ]:
coverage = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
    LIMIT 10
""").df()

print('Per-client history start dates differ — this is the unbalanced panel, shown directly:')
coverage


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself in Colab with your HF_TOKEN secret before committing**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support (`is_declining_proxy` is explicitly named a proxy, not a real outcome)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
